# Bound-Retention Model Training Demo

This notebook is a tidy demonstration version of the training flow in `scripts/train_bound_models.py`.

It focuses on the three classification models already used in this repo:
- Logistic regression
- Random forest
- Gradient boosting

The notebook shows how the data are prepared, how grouped cross-validation is evaluated, and how to save trained `.pkl` pipelines for later use.

## What this notebook demonstrates

- Uses the real bound-outcomes dataset from `outputs/bound_outcomes.csv`
- Reuses the same filename parsing and engineered features as the training script
- Compares grouped cross-validation performance across three classifiers
- Visualises ROC and confusion matrices
- Saves trained pipelines to disk

Default target: `bound_mass_fraction_ge_0_1`.

Why this target is useful: it turns the bound-retention problem into a coarse screening decision, which is a good match for ML triage. A threshold like 10% is easier to model robustly than a fully detailed fragment-by-fragment physical outcome.

You can switch to `has_any_bound_mass` below if you want the alternate classification task.

In [ ]:
from __future__ import annotations

import pickle
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
N_SPLITS = 5

DATASET_PATH = Path("outputs/bound_outcomes.csv")
SAVE_DIR = Path("ml/bound_outcomes/notebook_demo")

TARGET_NAME = "bound_mass_fraction_ge_0_1"  # change to "has_any_bound_mass" if needed
FEATURE_SET_NAME = "with_fof_linking_length"

FEATURE_SET_COLUMNS = {
    "with_fof_linking_length": [
        "mass_log10_kg",
        "particle_log10",
        "periapsis_Rm",
        "v_inf_kms",
        "spin_period_hr",
        "spin_axis",
        "has_explicit_spin",
        "special_case_code",
        "timestep",
        "fof_linking_length",
    ],
    "without_fof_linking_length": [
        "mass_log10_kg",
        "particle_log10",
        "periapsis_Rm",
        "v_inf_kms",
        "spin_period_hr",
        "spin_axis",
        "has_explicit_spin",
        "special_case_code",
        "timestep",
    ],
}

TARGET_LABELS = {
    "has_any_bound_mass": "Any bound mass retained?",
    "bound_mass_fraction_ge_0_1": "Bound mass fraction at least 0.1?",
}

SAVE_DIR.mkdir(parents=True, exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")

## Why these inputs are used

These features are chosen because they encode the simulation setup in a compact way that the model can learn from:
- `mass_log10_kg`, `periapsis_Rm`, and `v_inf_kms` capture the main encounter scale and geometry
- `spin_period_hr`, `spin_axis`, and `has_explicit_spin` capture rotational state
- `timestep`, `resolution`, and `fof_linking_length` capture numerical setup choices that can affect measured FoF outcomes

The training script parses these values from the filename because that is how the simulation campaign is structured in this repo. Reusing that logic here keeps the notebook consistent with the actual pipeline.

In [ ]:
FILENAME_RE = re.compile(
    r"^(?P<prefix>Ma_xp)_(?P<mass>A\d{4}(?:c30)?)(?:_(?P<spin>s\d{3}[A-Za-z]*))?"
    r"_n(?P<resolution>\d+)_r(?P<periapsis>\d+)_v(?P<velocity>\d+)"
    r"_(?P<timestep>\d+)"
    r"_fof_(?P<linking_length>[0-9.]+)_"
    r"(?P<chunk>\d+)\.hdf5$"
)


def parse_simulation_filename(filename: str) -> dict[str, object]:
    match = FILENAME_RE.match(filename)
    if not match:
        raise ValueError(f"Unrecognized FoF filename pattern: {filename}")

    mass_code = match.group("mass")
    spin_code = match.group("spin") or ""
    special_case_code = "c30" if mass_code.endswith("c30") else ""
    mass_digits = mass_code[1:5]
    spin_axis = spin_code[4:] if len(spin_code) > 4 else ""
    spin_value = spin_code[1:4] if spin_code else ""

    resolution_value = int(match.group("resolution"))
    periapsis_value = int(match.group("periapsis"))
    velocity_value = int(match.group("velocity"))
    timestep = int(match.group("timestep"))
    chunk_index = int(match.group("chunk"))
    linking_length = float(match.group("linking_length"))

    return {
        "filename": filename,
        "mass_code": mass_code,
        "mass_value": int(mass_digits),
        "special_case_code": special_case_code,
        "spin_code": spin_code,
        "spin_value": int(spin_value) if spin_value else "",
        "spin_axis": spin_axis,
        "has_explicit_spin": bool(spin_code),
        "resolution_code": f"n{resolution_value}",
        "resolution_value": resolution_value,
        "periapsis_value": periapsis_value,
        "velocity_value": velocity_value,
        "timestep": timestep,
        "fof_linking_length": linking_length,
        "chunk_index": chunk_index,
    }


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    frame = df.copy()
    parsed = frame["fof_file"].map(parse_simulation_filename).apply(pd.Series)
    for column in parsed.columns:
        if column not in frame.columns:
            frame[column] = parsed[column]

    frame["mass_log10_kg"] = pd.to_numeric(frame["mass_value"], errors="coerce") / 100.0
    resolution_values = pd.to_numeric(frame["resolution_value"], errors="coerce")
    frame["particle_log10"] = resolution_values.map(lambda x: np.nan if pd.isna(x) else np.log10(x))
    frame["periapsis_Rm"] = pd.to_numeric(frame["periapsis_value"], errors="coerce") / 10.0
    frame["v_inf_kms"] = pd.to_numeric(frame["velocity_value"], errors="coerce") / 10.0
    frame["spin_period_hr"] = pd.to_numeric(frame["spin_value"], errors="coerce") / 10.0
    frame["has_explicit_spin"] = (
        frame["has_explicit_spin"].fillna(False).astype(str).str.lower().isin({"true", "1", "yes"})
    )
    frame["spin_axis"] = frame["spin_axis"].fillna("none").replace("", "none")
    frame["special_case_code"] = frame["special_case_code"].fillna("").replace("", "none")
    frame["has_any_bound_mass"] = pd.to_numeric(frame["bound_mass_fraction"], errors="coerce") > 0
    frame["bound_mass_fraction_ge_0_1"] = pd.to_numeric(frame["bound_mass_fraction"], errors="coerce") >= 0.1
    bound_mass_kg = pd.to_numeric(frame["bound_mass_kg"], errors="coerce")
    bound_fragment_count = pd.to_numeric(frame["bound_fragment_count"], errors="coerce").replace(0, np.nan)
    frame["average_bound_fragment_mass_kg"] = bound_mass_kg / bound_fragment_count
    return frame

## Why these models are compared

- Logistic regression is the simplest baseline. It is fast, interpretable, and useful for checking whether the signal is mostly linear after preprocessing.
- Random forest is included because it can model nonlinear interactions and mixed feature types with relatively little tuning.
- Gradient boosting is included because it often performs well on structured tabular data when the signal is nonlinear but still fairly low-dimensional.

Comparing all three gives a reasonable tradeoff between interpretability, flexibility, and predictive power without making the demo too large.

In [ ]:
def build_preprocessor(X: pd.DataFrame, model_name: str) -> ColumnTransformer:
    categorical_features = [column for column in ["spin_axis", "special_case_code"] if column in X.columns]
    numeric_features = [column for column in X.columns if column not in categorical_features]

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if model_name == "logistic_regression":
        numeric_steps.append(("scaler", StandardScaler()))

    return ColumnTransformer(
        transformers=[
            ("num", Pipeline(numeric_steps), numeric_features),
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_features,
            ),
        ]
    )


def build_classifier_models(X: pd.DataFrame) -> dict[str, Pipeline]:
    return {
        "logistic_regression": Pipeline(
            [
                ("preprocessor", build_preprocessor(X, "logistic_regression")),
                ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
            ]
        ),
        "random_forest_classifier": Pipeline(
            [
                ("preprocessor", build_preprocessor(X, "random_forest_classifier")),
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=300,
                        min_samples_leaf=2,
                        class_weight="balanced_subsample",
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                    ),
                ),
            ]
        ),
        "gradient_boosting_classifier": Pipeline(
            [
                ("preprocessor", build_preprocessor(X, "gradient_boosting_classifier")),
                ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
            ]
        ),
    }


def classification_metric_summary(y_true: pd.Series, y_pred: np.ndarray, y_score: np.ndarray | None) -> dict[str, float]:
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }
    metrics["roc_auc"] = float(roc_auc_score(y_true, y_score)) if y_score is not None and y_true.nunique() > 1 else np.nan
    return metrics


def evaluate_classifier(name: str, base_pipeline: Pipeline, X: pd.DataFrame, y: pd.Series, groups: pd.Series) -> tuple[dict[str, float], pd.DataFrame, Pipeline]:
    splitter = GroupKFold(n_splits=min(N_SPLITS, groups.nunique()))
    y_pred = pd.Series(index=y.index, dtype="bool")
    y_score = pd.Series(index=y.index, dtype="float64")

    for train_idx, test_idx in splitter.split(X, y, groups):
        pipeline = clone(base_pipeline)
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train = y.iloc[train_idx]
        pipeline.fit(X_train, y_train)
        y_pred.loc[X_test.index] = pipeline.predict(X_test).astype(bool)
        y_score.loc[X_test.index] = pipeline.predict_proba(X_test)[:, 1]

    oof_scores = y_score.loc[y.index].to_numpy()
    test_metrics = classification_metric_summary(y, y_pred.loc[y.index].to_numpy(), oof_scores)

    final_pipeline = clone(base_pipeline)
    final_pipeline.fit(X, y)
    train_pred = final_pipeline.predict(X).astype(bool)
    train_score = final_pipeline.predict_proba(X)[:, 1]
    train_metrics = classification_metric_summary(y, train_pred, train_score)

    row = {
        "model": name,
        **test_metrics,
        **{f"train_{key}": value for key, value in train_metrics.items()},
    }
    prediction_frame = pd.DataFrame(
        {
            "actual": y.astype(bool),
            "predicted": y_pred.astype(bool),
            "score": y_score.astype(float),
        }
    )
    return row, prediction_frame, final_pipeline

## Why grouped cross-validation is used

The split is grouped by `physical_file` rather than by individual rows. That matters because multiple FoF runs can come from the same underlying physical simulation.

If those related rows were split across train and test sets, the model could look better than it really is by seeing nearly duplicate physical cases during training. Grouped CV is a stricter and more honest performance check for this dataset.

In [ ]:
raw_df = pd.read_csv(DATASET_PATH, low_memory=False)
df = add_engineered_features(raw_df)

feature_columns = FEATURE_SET_COLUMNS[FEATURE_SET_NAME]
X = df[feature_columns].copy()
y = df[TARGET_NAME].astype(bool)
groups = df["physical_file"].astype(str)

summary = pd.DataFrame(
    {
        "rows": [len(df)],
        "unique_physical_files": [groups.nunique()],
        "positive_rate": [y.mean()],
        "target": [TARGET_NAME],
        "target_label": [TARGET_LABELS[TARGET_NAME]],
    }
)
summary

In [ ]:
models = build_classifier_models(X)
metric_rows = []
prediction_frames = {}
trained_models = {}

for model_name, pipeline in models.items():
    metrics_row, predictions, fitted_pipeline = evaluate_classifier(model_name, pipeline, X, y, groups)
    metric_rows.append(metrics_row)
    prediction_frames[model_name] = predictions
    trained_models[model_name] = fitted_pipeline

metrics_df = pd.DataFrame(metric_rows).sort_values(["balanced_accuracy", "f1", "roc_auc"], ascending=[False, False, False])
metrics_df

## Performance plots

The plots below use out-of-fold predictions from grouped cross-validation, so they reflect held-out physical-file groups rather than in-sample fits.

Why these plots are shown:
- The confusion matrix shows the kinds of mistakes each classifier makes at the default decision threshold.
- The ROC curve shows how well the score separates positive and negative cases across thresholds.
- Looking at both is more informative than a single headline metric.

In [ ]:
fig, axes = plt.subplots(len(prediction_frames), 2, figsize=(12, 4 * len(prediction_frames)))
if len(prediction_frames) == 1:
    axes = np.array([axes])

for row_index, model_name in enumerate(metrics_df["model"]):
    prediction_frame = prediction_frames[model_name]
    ConfusionMatrixDisplay.from_predictions(
        prediction_frame["actual"],
        prediction_frame["predicted"],
        ax=axes[row_index, 0],
        colorbar=False,
    )
    axes[row_index, 0].set_title(f"{model_name} confusion matrix")

    RocCurveDisplay.from_predictions(
        prediction_frame["actual"],
        prediction_frame["score"],
        ax=axes[row_index, 1],
    )
    axes[row_index, 1].set_title(f"{model_name} ROC curve")

fig.suptitle(f"Grouped CV performance for {TARGET_LABELS[TARGET_NAME]}", fontsize=14, y=1.02)
fig.tight_layout()

In [ ]:
best_model_name = metrics_df.iloc[0]["model"]
best_model_name

## Save trained pipelines

This cell saves all three fitted pipelines as `.pkl` files. Each file includes the preprocessing steps and the trained classifier.

Why save the full pipeline instead of only the estimator: the preprocessing is part of the model definition. Saving the full pipeline ensures that the same imputing, scaling, and one-hot encoding are reused at prediction time.

In [ ]:
saved_paths = []
for model_name, fitted_pipeline in trained_models.items():
    output_path = SAVE_DIR / f"{TARGET_NAME}__{FEATURE_SET_NAME}__{model_name}.pkl"
    with output_path.open("wb") as handle:
        pickle.dump(fitted_pipeline, handle)
    saved_paths.append(str(output_path))

pd.DataFrame({"saved_model": saved_paths})

## Notes

- This notebook demonstrates the classification side of the repo in a compact format.
- The chosen classification tasks are appropriate for triage because they answer coarse screening questions, not full SPH-level physical detail.
- The wider repo also uses regression for bound-retention targets such as `bound_mass_fraction`, `bound_fragment_count`, `largest_bound_fragment_mass_kg`, and `average_bound_fragment_mass_kg`.
- If you want, the next cleanup step can be a second notebook focused on those regression targets and their performance plots.